# Step 5 — Curated Examples (Query, Filter, Measure, Field, Join)

**Mistplay Genie Workshop**

The metric views cover the *governed* measures. But real questions hit patterns they don't model — an **anti-join** (orders that never shipped), an **exception** (refunded-but-shipped), refunds as **negative** revenue.

You teach Genie these by adding **curated examples**. We add all five types, ~2 each:

| Type | What it does | Examples added |
|------|--------------|----------------|
| **Example query** | Full question→SQL the model can imitate | unshipped orders (anti-join); refunded-but-shipped |
| **Filter** | Reusable WHERE snippet | high-value orders; refunded only |
| **Measure** | Ad-hoc measure beyond the metric view | revenue net of refunds; distinct customers |
| **Field** | Computed/derived column | customer age group; order size bucket |
| **Join** | Relationship path | orders↔shipments; transactions↔shipments |

> **API vs UI:** example SQL, join specs, and the `sql_snippets` filters/expressions/measures are all set through the REST API. All in code — no UI step.

In [ ]:
import os, sys, importlib
nb_dir = os.getcwd()
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import workshop_lib.config, workshop_lib.genie_builder, workshop_lib.examples, workshop_lib.questions
for m in (workshop_lib.config, workshop_lib.genie_builder, workshop_lib.examples, workshop_lib.questions):
    importlib.reload(m)
from workshop_lib.config import WorkshopConfig, setup_widgets
from workshop_lib import genie_builder as gb
from workshop_lib import examples
from workshop_lib import questions
setup_widgets(dbutils)
cfg = WorkshopConfig.from_widgets(dbutils)
client = gb.GenieClient()
space_id = gb.require_space_id(spark, cfg)
print("space:", space_id)

## 1. Add the curated examples

`examples.add_examples(builder, cfg)` attaches all ten items. Open `workshop_lib/examples.py` to read each one — every example is a *high-quality* pattern not already defined in Steps 3-4.

In [ ]:
builder = client.load_builder(space_id)
counts = examples.add_examples(builder, cfg)
print("added:", counts)
print(builder.summary())

## 2. Patch the space

In [ ]:
client.patch_space(space_id, title=cfg.genie_space_name,
                   description="Mistplay Genie Workshop — with curated examples.",
                   warehouse_id=cfg.warehouse_id, serialized_space=builder.to_serialized())
print("Patched.")

## 3. Verify the example assets landed

In [ ]:
d = client.load_builder(space_id).to_dict()
instr = d.get("instructions", {})
snip = instr.get("sql_snippets", {})
print("example_question_sqls:", len(instr.get("example_question_sqls", [])))
print("join_specs           :", len(instr.get("join_specs", [])))
print("filters              :", len(snip.get("filters", [])))
print("expressions (fields) :", len(snip.get("expressions", [])))
print("measures             :", len(snip.get("measures", [])))

## 4. Try it — the two question sets

In [ ]:
print(questions.render(5))

## Recap & what's next

Genie now follows curated patterns for joins and exceptions. What's still missing is *business rules that aren't math*: the fiscal calendar, "exclude internal accounts by default", what "active customer" means, and answer tone.

**→ Step 6** adds **general instructions** — the natural-language layer.